In [11]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\MTA_Subway_Delay-Causing_Incidents__Beginning_2020_20260305.csv')

# Parse Month column (format: YYYY-MM-DD)
df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m-%d')

# Filter to rolling 12-month window: Feb 15 2025 to Feb 15 2026
df = df[(df['Month'] >= '2025-02-15') & (df['Month'] <= '2026-02-15')]

# Sum incidents by Line
line_totals = df.groupby('Line')['Incidents'].sum().reset_index()
line_totals.columns = ['line_id', 'total_incidents']

line_totals.to_csv(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_incidents_by_line.csv', index=False)
print(line_totals)

   line_id  total_incidents
0        1             3816
1        2             4945
2        3             2658
3        4             3861
4        5             2519
5        6             4787
6        7             2289
7        A             4223
8        B             1839
9        C             2335
10       D             3338
11       E             2891
12       F             4635
13       G             2194
14      JZ             2149
15       L             1968
16       M             1822
17       N             4481
18       Q             3571
19       R             2145
20  S 42nd               51
21  S Fkln               80
22  S Rock              655


In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString

gtfs_path = r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra'

# Load shapes and routes
shapes = pd.read_csv(f'{gtfs_path}\\shapes.txt')
routes = pd.read_csv(f'{gtfs_path}\\routes.txt')
trips  = pd.read_csv(f'{gtfs_path}\\trips.txt')

# Build shape_id -> route_short_name mapping
trip_map = trips[['route_id','shape_id']].drop_duplicates()
route_map = routes[['route_id','route_short_name']]
shape_to_line = trip_map.merge(route_map, on='route_id')[['shape_id','route_short_name']].drop_duplicates()

# Build polylines per shape_id
shapes_sorted = shapes.sort_values(['shape_id','shape_pt_sequence'])
lines = []
for shape_id, grp in shapes_sorted.groupby('shape_id'):
    coords = list(zip(grp['shape_pt_lon'], grp['shape_pt_lat']))
    if len(coords) >= 2:
        lines.append({'shape_id': shape_id, 'geometry': LineString(coords)})

gdf = gpd.GeoDataFrame(lines, crs='EPSG:4326')
gdf = gdf.merge(shape_to_line, on='shape_id', how='left')

# Dissolve to one polyline per line letter (route_short_name)
gdf_dissolved = gdf.dissolve(by='route_short_name').reset_index()
gdf_dissolved = gdf_dissolved.rename(columns={'route_short_name': 'line_id'})

out_path = f'{gtfs_path}\\mta_subway_lines.shp'
gdf_dissolved.to_file(out_path)
print(f'Saved: {out_path}')

Saved: C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.shp


In [3]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\MTA_Subway_Delay-Causing_Incidents__Beginning_2020_20260305.csv')

# Parse Month column
df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m-%d')

# Filter to rolling 12-month window
df = df[(df['Month'] >= '2025-02-15') & (df['Month'] <= '2026-02-15')]

# Sum incidents by Line
line_totals = df.groupby('Line')['Incidents'].sum().reset_index()
line_totals.columns = ['line_id', 'total_incidents']

# Convert line_id to string to match ArcGIS field type
line_totals['line_id'] = line_totals['line_id'].astype(str).str.strip()

print(line_totals)
print("\nUnique line_ids:", sorted(line_totals['line_id'].tolist()))

line_totals.to_csv(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_incidents_by_line.csv', index=False)
print("\nSaved.")

   line_id  total_incidents
0        1             3816
1        2             4945
2        3             2658
3        4             3861
4        5             2519
5        6             4787
6        7             2289
7        A             4223
8        B             1839
9        C             2335
10       D             3338
11       E             2891
12       F             4635
13       G             2194
14      JZ             2149
15       L             1968
16       M             1822
17       N             4481
18       Q             3571
19       R             2145
20  S 42nd               51
21  S Fkln               80
22  S Rock              655

Unique line_ids: ['1', '2', '3', '4', '5', '6', '7', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'JZ', 'L', 'M', 'N', 'Q', 'R', 'S 42nd', 'S Fkln', 'S Rock']

Saved.


In [5]:
import geopandas as gpd

gdf = gpd.read_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.shp')
print(sorted(gdf['line_id'].tolist()))

['1', '2', '3', '4', '5', '6', '6X', '7', '7X', 'A', 'B', 'C', 'D', 'E', 'F', 'FX', 'G', 'J', 'L', 'M', 'N', 'Q', 'R', 'S', 'SIR', 'W', 'Z']


In [8]:
import pandas as pd
import geopandas as gpd

# ── Load and filter incidents ──────────────────────────────────────────────────
df = pd.read_csv(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\MTA_Subway_Delay-Causing_Incidents__Beginning_2020_20260305.csv')
df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m-%d')
df = df[(df['Month'] >= '2025-02-15') & (df['Month'] <= '2026-02-15')]
line_totals = df.groupby('Line')['Incidents'].sum().reset_index()
line_totals.columns = ['line_id', 'total_incidents']
line_totals['line_id'] = line_totals['line_id'].astype(str).str.strip()

# ── Remap incidents line_ids to match shapefile ────────────────────────────────
# JZ splits equally between J and Z (they share the same physical route)
jz_count = line_totals.loc[line_totals['line_id'] == 'JZ', 'total_incidents'].values
if len(jz_count) > 0:
    half = jz_count[0] / 2
    line_totals = line_totals[line_totals['line_id'] != 'JZ']
    line_totals = pd.concat([
        line_totals,
        pd.DataFrame({'line_id': ['J', 'Z'], 'total_incidents': [half, half]})
    ], ignore_index=True)

# All S shuttle variants collapse to S
line_totals['line_id'] = line_totals['line_id'].replace({
    'S 42nd': 'S',
    'S Fkln': 'S',
    'S Rock': 'S'
})
line_totals = line_totals.groupby('line_id')['total_incidents'].sum().reset_index()

# ── Merge onto shapefile and export ───────────────────────────────────────────
gdf = gpd.read_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.shp')
gdf = gdf.merge(line_totals, on='line_id', how='left')
gdf['total_incidents'] = gdf['total_incidents'].fillna(0)

# Verify — print every line and its incident count
print(gdf[['line_id', 'total_incidents']].sort_values('line_id').to_string())

# Save back to shapefile (overwrites mta_subway_lines.shp with incidents attached)
gdf.to_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.shp')
print("\nDone. Shapefile updated with total_incidents column.")

   line_id  total_incidents
0        1           3816.0
1        2           4945.0
2        3           2658.0
3        4           3861.0
4        5           2519.0
5        6           4787.0
6       6X              0.0
7        7           2289.0
8       7X              0.0
9        A           4223.0
10       B           1839.0
11       C           2335.0
12       D           3338.0
13       E           2891.0
14       F           4635.0
15      FX              0.0
16       G           2194.0
17       J           1074.5
18       L           1968.0
19       M           1822.0
20       N           4481.0
21       Q           3571.0
22       R           2145.0
23       S            786.0
24     SIR              0.0
25       W              0.0
26       Z           1074.5

Done. Shapefile updated with total_incidents column.


C:\Users\Arv\AppData\Local\Temp\ipykernel_24416\3422648550.py:40: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.shp')
c:\Users\Arv\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'total_incidents' to 'total_in_1'
  ogr_write(


In [14]:
import pandas as pd
import geopandas as gpd

# ── Load and filter incidents ──────────────────────────────────────────────────
df = pd.read_csv(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\MTA_Subway_Delay-Causing_Incidents__Beginning_2020_20260305.csv')
df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m-%d')
df = df[(df['Month'] >= '2025-02-15') & (df['Month'] <= '2026-02-15')]
line_totals = df.groupby('Line')['Incidents'].sum().reset_index()
line_totals.columns = ['line_id', 'total_incidents']
line_totals['line_id'] = line_totals['line_id'].astype(str).str.strip()

# ── Remap incidents line_ids to match shapefile ────────────────────────────────
jz_count = line_totals.loc[line_totals['line_id'] == 'JZ', 'total_incidents'].values
if len(jz_count) > 0:
    half = jz_count[0] / 2
    line_totals = line_totals[line_totals['line_id'] != 'JZ']
    line_totals = pd.concat([
        line_totals,
        pd.DataFrame({'line_id': ['J', 'Z'], 'total_incidents': [half, half]})
    ], ignore_index=True)

line_totals['line_id'] = line_totals['line_id'].replace({
    'S 42nd': 'S',
    'S Fkln': 'S',
    'S Rock': 'S'
})
line_totals = line_totals.groupby('line_id')['total_incidents'].sum().reset_index()

# ── Read geometry-only from overwritten shapefile, drop the truncated column ──
gdf = gpd.read_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.shp')
gdf = gdf.drop(columns=['total_in_1'], errors='ignore')  # drop the bad column

# Re-merge with clean incident counts
gdf = gdf.merge(line_totals, on='line_id', how='left')
gdf['total_incidents'] = gdf['total_incidents'].fillna(0)

# Verify
print(gdf[['line_id', 'total_incidents']].sort_values('line_id').to_string())

# ── Save as GeoPackage ────────────────────────────────────────────────────────
out_path = r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.gpkg'
gdf.to_file(out_path, driver='GPKG')
print("\nSaved. No column truncation warnings should appear above.")

   line_id  total_incidents
0        1           3816.0
1        2           4945.0
2        3           2658.0
3        4           3861.0
4        5           2519.0
5        6           4787.0
6       6X              0.0
7        7           2289.0
8       7X              0.0
9        A           4223.0
10       B           1839.0
11       C           2335.0
12       D           3338.0
13       E           2891.0
14       F           4635.0
15      FX              0.0
16       G           2194.0
17       J           1074.5
18       L           1968.0
19       M           1822.0
20       N           4481.0
21       Q           3571.0
22       R           2145.0
23       S            786.0
24     SIR              0.0
25       W              0.0
26       Z           1074.5

Saved. No column truncation warnings should appear above.


In [3]:
import os
import geopandas as gpd

gdf = gpd.read_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_subway_lines.gpkg')

os.makedirs(r'C:\GIS_temp', exist_ok=True)
out_path = r'C:\GIS_temp\mta_subway_lines.gpkg'
gdf.to_file(out_path, driver='GPKG')
print("Saved to C:\\GIS_temp\\mta_subway_lines.gpkg")

Saved to C:\GIS_temp\mta_subway_lines.gpkg


In [15]:
import os
import pandas as pd
import geopandas as gpd

# ── Load intersection CSV ──────────────────────────────────────────────────────
df = pd.read_csv(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\mta_routes_nta_intersect.csv')

# For each line, compute total route length across all NTAs
df['line_total_km'] = df.groupby('line_id')['seg_length_km'].transform('sum')

# Weight = this segment's length / line's total length
df['weight'] = df['seg_length_km'] / df['line_total_km']

# Allocate incidents proportionally
df['alloc_incidents'] = df['total_incidents'] * df['weight']

# Sum allocated incidents per NTA across all lines
nta = df.groupby('ntaabbrev')['alloc_incidents'].sum().reset_index()
nta.columns = ['ntaabbrev', 'alloc_incidents']

# ── Compute area_km2 from NTA geojson ─────────────────────────────────────────
nta_gdf = gpd.read_file(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\2020_Neighborhood_Tabulation_Areas_(NTAs)_20260305.geojson')
nta_gdf = nta_gdf.to_crs(epsg=32618)  # UTM zone 18N for accurate area in metres
nta_gdf['area_km2'] = nta_gdf.geometry.area / 1e6
areas = nta_gdf[['ntaabbrev', 'area_km2']]

# ── Merge and compute density ──────────────────────────────────────────────────
nta = nta.merge(areas, on='ntaabbrev', how='right')
nta['alloc_incidents'] = nta['alloc_incidents'].fillna(0)
nta['mta_delay_per_km2'] = nta['alloc_incidents'] / nta['area_km2']

# ── Save ───────────────────────────────────────────────────────────────────────
os.makedirs(r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\arcgis_exports', exist_ok=True)
nta[['ntaabbrev', 'alloc_incidents', 'mta_delay_per_km2']].to_csv(
    r'C:\Users\Arv\OneDrive\Desktop\arcgisnycinfra\arcgis_exports\nta_mta_delays.csv',
    index=False
)
print(nta[['ntaabbrev', 'mta_delay_per_km2']].sort_values('mta_delay_per_km2', ascending=False).head(10))

      ntaabbrev  mta_delay_per_km2
28     LclnTrPk        1324.050865
122   GrnwchVlg        1218.673032
130  Mdtwn_TmSq        1153.179967
129    MdtwnSth        1079.662609
121        SoHo        1068.796642
119     TriBeCa        1025.101641
5      DwntwnBk         903.925917
162  SnnysdYd_N         780.417019
140  UES_CrngHl         773.366921
123      WstVlg         646.940917
